# UK Cross-Asset Market Monitor

This notebook tracks weekly moves across key UK and global market indicators, including gilt yields, UK equities, FX, commodities and global risk sentiment indicators.

In [35]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import matplotlib.ticker as mtick

In [36]:
START_DATE = '2025-01-01'
END_DATE = None

ASSETS = {'FTSE100': '^FTSE', 'FTSE250': '^FTMC', 'GBP/USD': 'GBPUSD=X', 'EUR/GBP': 'EURGBP=X', 'Brent Crude': 'BZ=F', 'S&P500': '^GSPC', 'VIX': '^VIX', 'US 10Y Treasury Yield': '^TNX'}


# Market Data Download

This section downloads daily market data for the selected cross-asset indicators.

In [37]:
tickers = list(ASSETS.values())

market_data = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)

open_prices = market_data['Open'].rename(columns={v: k for k, v in ASSETS.items()})
close_prices = market_data['Close'].rename(columns={v: k for k, v in ASSETS.items()})
open_prices = open_prices.dropna(how='all')
close_prices = close_prices.dropna(how='all')

close_prices.tail()

[*********************100%***********************]  8 of 8 completed


Ticker,Brent Crude,EUR/GBP,GBP/USD,FTSE250,FTSE100,S&P500,US 10Y Treasury Yield,VIX
Date,,,,,,,,
2026-07-08,78.019997,0.85423,1.334891,23017.599609,10489.000000,7482.709961,4.569,16.900000
2026-07-09,76.300003,0.85249,1.339621,23240.599609,10472.500000,7543.640137,4.539,15.840000
2026-07-10,76.010002,0.85232,1.341562,23371.400391,10497.299805,7575.390137,4.569,15.030000
2026-07-13,83.300003,0.85190,1.338688,23396.599609,10498.299805,7515.339844,4.609,17.160000
2026-07-14,86.870003,0.85199,1.337811,23230.580078,10454.669922,NaN,NaN,17.290001


In [38]:
daily_returns = close_prices.pct_change().dropna(how='all')
daily_returns.tail()

Ticker,Brent Crude,EUR/GBP,GBP/USD,FTSE250,FTSE100,S&P500,US 10Y Treasury Yield,VIX
Date,,,,,,,,
2026-07-08,0.052050,0.000059,-0.003656,-0.015450,-0.016586,-0.002817,0.008832,0.047737
2026-07-09,-0.022046,-0.002037,0.003543,0.009688,-0.001573,0.008143,-0.006566,-0.062722
2026-07-10,-0.003801,-0.000199,0.001449,0.005628,0.002368,0.004209,0.006609,-0.051136
2026-07-13,0.095908,-0.000493,-0.002142,0.001078,0.000095,-0.007927,0.008755,0.141717
2026-07-14,0.042857,0.000106,-0.000656,-0.007096,-0.004156,NaN,NaN,0.007576


In [39]:
def create_weekly_summary_for_week(selected_date = None):
    if selected_date is None:
        latest_date = close_prices.dropna(how="all").index[-1]
    else:
        latest_date = pd.to_datetime(selected_date)

    latest_friday = latest_date - pd.Timedelta(days=(latest_date.weekday() - 4) % 7)
    week_start = latest_friday - pd.Timedelta(days=4)

    weekly_open_prices = open_prices.loc[(open_prices.index >= week_start) & (open_prices.index <= latest_friday)]

    weekly_close_prices = close_prices.loc[(close_prices.index >= week_start) & (close_prices.index <= latest_friday)]

    monday_open = weekly_open_prices.ffill().iloc[0]
    friday_close = weekly_close_prices.ffill().iloc[-1]

    weekly_change = friday_close - monday_open
    weekly_pct_change = (friday_close - monday_open) / monday_open

    weekly_summary = pd.DataFrame({ "Monday Open": monday_open, "Friday Close": friday_close, "Weekly Change": weekly_change, "Weekly % Change": weekly_pct_change })

    yield_assets = ['US 10Y Treasury Yield']
    weekly_summary['Change (bps)'] = np.nan
    for asset in yield_assets:
        weekly_summary.loc[asset, 'Change (bps)'] = weekly_summary.loc[asset, 'Weekly Change'] * 100
        weekly_summary.loc[asset, 'Weekly Change'] = np.nan
        weekly_summary.loc[asset, 'Weekly % Change'] = np.nan

    

    weekly_summary["Weekly % Change"] = weekly_summary["Weekly % Change"].map(lambda x:"" if pd.isna(x) else f"{x:.2%}")
    weekly_summary["Change (bps)"] = weekly_summary["Change (bps)"].map(lambda x:"" if pd.isna(x) else f"{x:.1f} bps")
    weekly_summary["Weekly Change"] = weekly_summary["Weekly Change"].map(lambda x:"" if pd.isna(x) else x)

    return weekly_summary, week_start, latest_friday

weekly_summary, week_start, latest_friday = create_weekly_summary_for_week()

weekly_summary

,Monday Open,Friday Close,Weekly Change,Weekly % Change,Change (bps)
Ticker,,,,,
Brent Crude,71.870003,76.010002,4.139999,5.76%,
EUR/GBP,0.856550,0.852320,-0.00423,-0.49%,
GBP/USD,1.335096,1.341562,0.006466,0.48%,
FTSE250,23539.099609,23371.400391,-167.699219,-0.71%,
FTSE100,10679.400391,10497.299805,-182.100586,-1.71%,
S&P500,7506.959961,7575.390137,68.430176,0.91%,
US 10Y Treasury Yield,4.457000,4.569000,,,11.2 bps
VIX,16.400000,15.030000,-1.37,-8.35%,


In [41]:
create_weekly_summary_for_week()

(                        Monday Open  Friday Close Weekly Change  \
 Ticker                                                            
 Brent Crude               71.870003     76.010002      4.139999   
 EUR/GBP                    0.856550      0.852320      -0.00423   
 GBP/USD                    1.335096      1.341562      0.006466   
 FTSE250                23539.099609  23371.400391   -167.699219   
 FTSE100                10679.400391  10497.299805   -182.100586   
 S&P500                  7506.959961   7575.390137     68.430176   
 US 10Y Treasury Yield      4.457000      4.569000                 
 VIX                       16.400000     15.030000         -1.37   
 
                       Weekly % Change Change (bps)  
 Ticker                                              
 Brent Crude                     5.76%               
 EUR/GBP                        -0.49%               
 GBP/USD                         0.48%               
 FTSE250                        -0.71%          

# Export Weekly Snapshot to Excel

This section saves the latest weekly market snapshot to a single Excel workbook. Each time the notebook is run for a new week, the latest weekly data is appended to the historical tracking file.

In [42]:
from pathlib import Path

def export_week_to_excel(selected_date=None):
    EXCEL_OUTPUT_FILE = "uk_market_monitor_history_clean.xlsx"

    weekly_summary, week_start, latest_friday = create_weekly_summary_for_week(selected_date)

    weekly_export = weekly_summary.copy()

    weekly_export.insert(0, "Asset", weekly_export.index)
    weekly_export.insert(0, "Week End", latest_friday.date())
    weekly_export.insert(0, "Week Start", week_start.date())

    weekly_export = weekly_export.reset_index(drop=True)

    excel_path = Path(EXCEL_OUTPUT_FILE)

    if excel_path.exists():
        existing_history = pd.read_excel(EXCEL_OUTPUT_FILE, sheet_name="Weekly History")

        updated_history = pd.concat(
            [existing_history, weekly_export],
            ignore_index=True
        )

        updated_history = updated_history.drop_duplicates(
            subset=["Week Start", "Week End", "Asset"],
            keep="last"
        )
    else:
        updated_history = weekly_export

    with pd.ExcelWriter(EXCEL_OUTPUT_FILE, engine="openpyxl", mode="w") as writer:
        updated_history.to_excel(writer, sheet_name="Weekly History", index=False)

    print(f"Saved week ending {latest_friday.date()} to {EXCEL_OUTPUT_FILE}")

    return weekly_summary

In [43]:
export_week_to_excel("2026-07-03")

Saved week ending 2026-07-03 to uk_market_monitor_history_clean.xlsx


,Monday Open,Friday Close,Weekly Change,Weekly % Change,Change (bps)
Ticker,,,,,
Brent Crude,73.169998,71.800003,-1.369995,-1.87%,
EUR/GBP,0.862720,0.856100,-0.00662,-0.77%,
GBP/USD,1.319609,1.333870,0.014261,1.08%,
FTSE250,23147.000000,23538.800781,391.800781,1.69%,
FTSE100,10507.799805,10679.000000,171.200195,1.63%,
S&P500,7391.879883,7483.240234,91.360352,1.24%,
US 10Y Treasury Yield,4.378000,4.485000,,,10.7 bps
VIX,18.600000,16.150000,-2.450001,-13.17%,


In [45]:
export_week_to_excel()

Saved week ending 2026-07-10 to uk_market_monitor_history_clean.xlsx


,Monday Open,Friday Close,Weekly Change,Weekly % Change,Change (bps)
Ticker,,,,,
Brent Crude,71.870003,76.010002,4.139999,5.76%,
EUR/GBP,0.856550,0.852320,-0.00423,-0.49%,
GBP/USD,1.335096,1.341562,0.006466,0.48%,
FTSE250,23539.099609,23371.400391,-167.699219,-0.71%,
FTSE100,10679.400391,10497.299805,-182.100586,-1.71%,
S&P500,7506.959961,7575.390137,68.430176,0.91%,
US 10Y Treasury Yield,4.457000,4.569000,,,11.2 bps
VIX,16.400000,15.030000,-1.37,-8.35%,
